# 🔬 Notebook 3: Discord — Deep Dive: Fan-out, Presence, Voice


## 🛠️ Setup

```bash
cd 06-system-designs/discord
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

All code in this lab is **self-contained Python** — no servers, no databases. We simulate Discord's gateway, pub/sub, and fan-out in memory so you can run every cell and see what happens.


## 💡 What we'll build

Three hard problems, each with a **bad → good → best** progression we can run:

1. **Fan-out** — how does one message reach 100 000 people?
2. **Presence** — how do we keep track of who's online without flooding the network?
3. **Voice** — why it uses UDP+WebRTC, not WebSockets.
4. **Rate limiting** — protecting a channel from spam.


## 📣 Fan-out — bad → best

You post one message. It has to reach every subscriber, which can be tens of thousands of sockets scattered across hundreds of gateway servers. Three ways to do it:

**❌ Bad — single server holds everyone.** Doesn't scale past ~50k connections.
**🙂 Good — every gateway gets every message, filters locally.** Simple, but wastes CPU: most messages aren't for most gateways.
**✅ Best — pub/sub by channel topic.** Only gateways that have *at least one* member of that channel subscribe to its topic.


In [ ]:
# ==== Fan-out comparison ====
import time, random
from collections import defaultdict
random.seed(0)

NUM_GATEWAYS = 500
NUM_CHANNELS = 5000
NUM_USERS = 50_000
CHANNELS_PER_USER = 3

# Assign each user to one gateway and to a few channels.
user_gateway = [random.randrange(NUM_GATEWAYS) for _ in range(NUM_USERS)]
channel_members = defaultdict(list)   # channel_id -> [user_id]
for u in range(NUM_USERS):
    for c in random.sample(range(NUM_CHANNELS), k=CHANNELS_PER_USER):
        channel_members[c].append(u)

# ---------- 🙂 GOOD: broadcast to every gateway, filter locally ----------
# Each gateway scans its members on every event — O(NUM_GATEWAYS × local_users) per msg.
# Build per-gateway subscriptions (channel -> users) for filtered delivery later.
gw_subs = [defaultdict(list) for _ in range(NUM_GATEWAYS)]
for c, members in channel_members.items():
    for u in members:
        gw_subs[user_gateway[u]][c].append(u)

def fanout_broadcast(channel_id):
    delivered = 0
    for g in range(NUM_GATEWAYS):           # every gateway sees every message
        for u in gw_subs[g].get(channel_id, ()):
            delivered += 1                  # pretend to send over the socket
    return delivered

# ---------- ✅ BEST: pub/sub by channel topic ----------
# Build an index: channel -> set of gateways that actually hold a subscriber.
channel_to_gateways = defaultdict(set)
for c, members in channel_members.items():
    for u in members:
        channel_to_gateways[c].add(user_gateway[u])

def fanout_topic(channel_id):
    delivered = 0
    for g in channel_to_gateways[channel_id]:   # only interested gateways
        for u in gw_subs[g].get(channel_id, ()):
            delivered += 1
    return delivered

# Benchmark: send 2000 messages across random channels
msgs = random.choices(list(channel_members.keys()), k=2000)

t0 = time.perf_counter(); total=0
for c in msgs: total += fanout_broadcast(c)
t_broadcast = time.perf_counter() - t0

t0 = time.perf_counter(); total2=0
for c in msgs: total2 += fanout_topic(c)
t_topic = time.perf_counter() - t0

print(f'🙂 GOOD  broadcast:  {t_broadcast*1000:6.1f} ms   delivered={total}')
print(f'✅ BEST  pub/sub:    {t_topic*1000:6.1f} ms   delivered={total2}')
print(f'Speedup from skipping uninterested gateways: {t_broadcast/t_topic:.1f}×')


**The lesson:** the number of *deliveries* is the same either way (the audience hasn't changed). What differs is the number of gateways that even *touch* the event. In production that difference is the gap between 300 Gbps of wasted cross-datacenter traffic and a comfortable 5 Gbps.


## 🐘 Fan-out to a *large* server — where the previous answer stops working

The demo above had ~30 members per channel, and pub/sub-by-topic looked like a clean win.
Discord's actual hard case is the opposite: a guild with **100,000+ members**, where one
`#announcements` post has to reach a large fraction of the entire gateway fleet.

Two things break at that size, and they break for different reasons. Let's measure both
instead of asserting them.

In [ ]:
# ==== Where does "only subscribe interested gateways" stop helping? ====
import random
random.seed(11)

GATEWAYS = 500

def gateways_touched(channel_size, gateways=GATEWAYS, trials=5):
    """Expected number of distinct gateways holding at least one member of the channel."""
    total = 0
    for _ in range(trials):
        total += len({random.randrange(gateways) for _ in range(channel_size)})
    return total / trials

print(f"{'channel members':>16} | {'gateways touched':>17} | {'% of fleet':>10} | pub/sub still helping?")
print("-" * 76)
for size in (5, 20, 50, 200, 1_000, 5_000, 20_000, 100_000):
    touched = gateways_touched(size)
    pct = touched / GATEWAYS
    verdict = "yes, hugely" if pct < 0.2 else ("marginal" if pct < 0.8 else "NO -- it IS a broadcast")
    print(f"{size:>16,} | {touched:>17,.0f} | {pct:>9.0%} | {verdict}")

print(f"""
The curve saturates fast. Past a few thousand members, essentially every gateway holds
someone, so 'publish only to interested gateways' degenerates into 'publish to everyone'.
Topic pub/sub is a filter, and there is nothing left to filter.""")

### Fix #1 — stop counting gateways, start counting *bytes on the bus*

Once every gateway is subscribed anyway, the remaining question is **how many copies of the
message cross the network**. The naive answer — the publisher sends one copy per recipient
socket — is the thing that actually melts, and it has nothing to do with pub/sub topics.

In [ ]:
# ==== 100k-member channel: three ways to get one message to everyone ====
MEMBERS   = 100_000
MSG_BYTES = 250
member_gateway = [random.randrange(GATEWAYS) for _ in range(MEMBERS)]

from collections import Counter
per_gateway = Counter(member_gateway)          # gateway -> how many of our members it holds

# ❌ NAIVE: the message service addresses each recipient individually.
naive_bus_msgs = MEMBERS

# ✅ BATCHED: publish ONE copy per gateway; each gateway expands it to its local sockets.
#    The socket writes still happen -- but they happen on the gateway's own loopback,
#    not across the datacenter fabric.
batched_bus_msgs = len(per_gateway)

# 🏆 LAZY SUBSCRIPTIONS: a client is only subscribed to channels it currently has OPEN.
#    In a 100k-member guild, almost nobody is looking at #announcements right now.
VIEWING = 0.02                                  # 2% have the channel focused
watchers = [g for g in member_gateway if random.random() < VIEWING]
lazy_gateways = len(set(watchers))
lazy_bus_msgs = lazy_gateways

for name, bus_msgs, sockets in [
    ("naive  (one bus msg per member)", naive_bus_msgs, MEMBERS),
    ("batched(one bus msg per gateway)", batched_bus_msgs, MEMBERS),
    ("lazy   (only open channels)     ", lazy_bus_msgs, len(watchers)),
]:
    print(f"{name}: bus messages={bus_msgs:>7,}  "
          f"bus bytes={bus_msgs*MSG_BYTES/1e6:>7.2f} MB  socket writes={sockets:>7,}")

print(f"""
batching cuts cross-datacenter traffic {naive_bus_msgs/batched_bus_msgs:.0f}x -- and it costs
nothing but a per-gateway loop that was going to run anyway.

lazy subscriptions cut the *socket writes* {MEMBERS/max(len(watchers),1):.0f}x, which is the
CPU and the client battery. It is the only one of the three that reduces actual delivery.""")

### Fix #2 — the member list is a bigger problem than the messages

The thing that actually broke Discord at 100k+ members was not `MESSAGE_CREATE`; it was the
**member list sidebar**. Sending 100,000 member objects (with presence, roles, and colours)
to every client that opens the guild is:

- a multi-megabyte payload per client, and
- a payload that must be *re-sent or patched* every time anyone's presence flips.

Their fix is worth remembering because it is a UI observation, not a distributed-systems one:
**the client can only display ~100 rows at a time**. So the server keeps the member list as
a sorted structure and ships only the *visible window*, plus deltas as the user scrolls.
100k members becomes 100 rows.

⚖️ **What all of this costs:**

| Technique | Buys | Costs |
|---|---|---|
| Per-gateway batching | ~200× less bus traffic | the gateway now owns the expand loop; a slow socket in that loop head-of-line-blocks the rest, so you need per-socket queues with drop policies |
| Lazy subscriptions | ~50× fewer socket writes | subscribe/unsubscribe churn on every UI navigation; the server must track *who is looking at what*, which is per-connection state on a supposedly stateless gateway |
| Windowed member list | megabytes → kilobytes | a sorted-set data structure per guild that must stay consistent under joins, leaves, role changes and presence flips — genuinely hard, and it is why this shipped years after the rest |
| A dedicated "guild process" per large guild | one place that owns fan-out, so ordering is trivial | a single-threaded hot spot, and a failover story for a process holding 100k members' state |

## 👀 Presence — the surprisingly hard problem

*"When Alice comes online, tell everyone who cares."* Sounds simple. It isn't:

- Alice has 200 friends and is in 30 guilds with 100k members total.
- Presence flips rapidly (tab sleeps → away, mouse moves → online).
- Doing it naively: every flip = 100 000 writes.

**❌ Bad:** broadcast every presence change to every member of every shared guild.
**✅ Best — lazy presence:** a user only receives presence events for people they **actually see right now** (open member list, DM list, mutual voice channel). Subscribing/unsubscribing happens when UI opens/closes.


In [ ]:
# ==== Presence: naive vs lazy ====
from collections import defaultdict

class NaivePresence:
    """Every guild member gets every flip."""
    def __init__(self, guild_members):
        self.guild_members = guild_members
        self.sent = 0
    def on_flip(self, user_id, guild_id):
        for _member in self.guild_members[guild_id]:
            self.sent += 1      # pretend to push over WS

class LazyPresence:
    """Only subscribers who have the user visible get the flip."""
    def __init__(self):
        self.watchers: dict[int, set[int]] = defaultdict(set)
        self.sent = 0
    def subscribe(self, watcher_id, target_id): self.watchers[target_id].add(watcher_id)
    def unsubscribe(self, watcher_id, target_id): self.watchers[target_id].discard(watcher_id)
    def on_flip(self, user_id):
        for _w in self.watchers[user_id]:
            self.sent += 1

# Setup: 1000 users, each in a big guild of 10 000 members
N_USERS = 1000
guild_members = {0: list(range(10_000))}
naive = NaivePresence(guild_members)
lazy = LazyPresence()

# In the 'lazy' world, each user only watches their ~50 friends
random.seed(1)
for u in range(N_USERS):
    for friend in random.sample(range(10_000), 50):
        lazy.subscribe(u, friend)

# 10 000 presence flips
for _ in range(10_000):
    target = random.randrange(10_000)
    naive.on_flip(target, 0)
    lazy.on_flip(target)

print(f'❌ naive presence pushes:  {naive.sent:>10,}')
print(f'✅ lazy  presence pushes:  {lazy.sent:>10,}')
print(f'Reduction: {naive.sent / max(lazy.sent,1):.0f}× fewer events to send.')


## 🎚️ Rate limiting per channel

Without a rate limiter, one bot could spam 10 000 messages/sec and force 10 000 × 50 = 500 000 events/sec of fan-out, starving real users. The classic fix is a **token bucket** per (user, channel).


In [ ]:
# ==== Token bucket rate limiter ====
import time

class TokenBucket:
    def __init__(self, capacity, refill_per_sec):
        self.capacity = capacity
        self.tokens = capacity
        self.refill = refill_per_sec
        self.last = time.monotonic()

    def allow(self, cost=1):
        now = time.monotonic()
        # refill based on elapsed time
        self.tokens = min(self.capacity, self.tokens + (now - self.last) * self.refill)
        self.last = now
        if self.tokens >= cost:
            self.tokens -= cost
            return True
        return False

# Discord-like: 5 messages per 5 seconds per (user, channel)
bucket = TokenBucket(capacity=5, refill_per_sec=1.0)

allowed = 0; rejected = 0
for i in range(20):
    if bucket.allow():
        allowed += 1
    else:
        rejected += 1
    time.sleep(0.1)      # send a message every 100ms

print(f'In 20 rapid attempts: {allowed} allowed, {rejected} rejected.')
print('With a 5-token bucket refilling at 1/s, burst of 5 is fine, then 1 per second.')


## 🎙️ Voice — why WebRTC and UDP

Text chat uses a WebSocket over TCP. Voice doesn't — and the reason is physics:

- A lost packet on TCP triggers a **retransmit**. Meanwhile the receiver buffers and waits. Late audio = stutter.
- For voice, **dropping a packet is better than pausing** — Opus can interpolate a missing 20 ms.
- So voice rides on **UDP**, via **WebRTC**, encoded with **Opus**.

Below we simulate how TCP-style 'wait for the missing packet' creates stutter compared to UDP-style 'skip it and keep going'.


In [ ]:
# ==== Stutter simulation: TCP-ish vs UDP-ish ====
import random
random.seed(2)

LOSS = 0.05   # 5% packet loss
RTT_MS = 60   # retransmit round-trip time
PKT_MS = 20   # one audio frame = 20 ms
NUM = 500

def simulate_tcp():
    playtime = 0
    for _ in range(NUM):
        while random.random() < LOSS:
            playtime += RTT_MS      # wait for retransmit — user hears silence
        playtime += PKT_MS
    return playtime

def simulate_udp():
    playtime = 0
    heard = 0
    for _ in range(NUM):
        playtime += PKT_MS           # always advance — silence or interpolated sample
        if random.random() >= LOSS:
            heard += 1
    return playtime, heard

tcp_ms = simulate_tcp()
udp_ms, udp_heard = simulate_udp()
print(f'🔁 TCP-style: playback took {tcp_ms} ms for {NUM*PKT_MS} ms of audio  → {tcp_ms/(NUM*PKT_MS):.2f}× real-time (stutter!)')
print(f'📡 UDP-style: playback took {udp_ms} ms  → 1.00× real-time, heard {udp_heard}/{NUM} frames')
print('\nFor voice, *steady rhythm* matters more than perfect delivery.')


### Selective Forwarding Unit (SFU)

A voice channel with 20 speakers and 1000 listeners can't have every listener connect to every speaker (N² connections). Instead, each speaker uploads their stream to an **SFU** — a server that forwards streams to listeners without re-encoding.

```
    speakers ──▶ SFU ──▶ listeners
     (20)              (1000)
```

- No transcoding ⇒ low CPU, sub-50 ms added latency.
- SFUs are regional — the client connects to the closest one.


## 🧩 Putting it all together

When Alice types "hi" in `#general`:

1. Client sends `POST /channels/{id}/messages` *and* emits via WebSocket with a `nonce`.
2. The API assigns a **Snowflake ID**, dedupes by nonce, writes to Cassandra (partition = channel_id).
3. It publishes `MESSAGE_CREATE` to the **channel topic** on Kafka.
4. Every gateway subscribed to that topic receives it exactly once.
5. Each gateway pushes the event to the sockets it holds for that channel.
6. Offline users → enqueue for push notifications.
7. Server increments the session's `s`; clients ack via heartbeats carrying the last `s` they saw.

## 🏁 Exercises

1. Change `NUM_GATEWAYS` in Notebook 1 and observe how the skew of random/modulo/consistent compares.
2. In the large-guild demo, change `VIEWING` from 2% to 50% (an active raid channel, not #announcements). At what point does lazy subscription stop paying for its bookkeeping?
3. Add a **priority** field to the rate limiter so that moderators get more tokens than regular users.
4. Extend the resume buffer to **evict events older than 3 min** (time-based, not just size-based).

## 📚 Further reading

- [How Discord handles 2 million concurrent voice users](https://discord.com/blog/how-discord-handles-two-and-half-million-concurrent-voice-users-using-webrtc)
- [How Discord stores billions of messages](https://discord.com/blog/how-discord-stores-billions-of-messages)
- [Twitter Snowflake](https://en.wikipedia.org/wiki/Snowflake_ID)
